# WAF Performance Efficiency
- Assessment Audit

This notebook assesses the Performance Efficiency pillar using the **4 Key Topics** structure.

## Scoring Model (1-5 Scale)
| Score | Rating | Description |
|-------|--------|-------------|
| 5 | Excellent | Best practices fully implemented |
| 4 | Good | Most best practices, minor gaps |
| 3 | Moderate | Basic implementation |
| 2 | Limited | Minimal implementation |
| 1 | Poor | Not implemented |

## 4 Key Topics

1. **Maximize**
- Query Optimization, Clustering, Materialized Views, Result Caching

2. **Scale**
- Warehouse Sizing, Multi-Cluster Warehouses, Workload Isolation, Scaling Policies

3. **Improve**
- Query Profiling, Query History Analysis, Warehouse Utilization, Iterative Tuning

4. **Leverage**
- Search Optimization Service, Unistore, Snowpark, Dynamic Tables

In [1]:
SCHEMA = "IE"
ACCOUNT_ID = 100058
DATABASE = "SNOWHOUSE_IMPORT"
print(f"Target: {DATABASE}.{SCHEMA} | Account ID: {ACCOUNT_ID}")

Target: SNOWHOUSE_IMPORT.IE | Account ID: 100058


In [2]:
from snowflake.snowpark import Session
import pandas as pd
import os

connection_name = os.getenv("SNOWFLAKE_CONNECTION_NAME") or "snowhouse"
if 'session' not in globals():
    session = Session.builder.config("connection_name", connection_name).create()
print(f"Connected via: {connection_name}")

/Users/edscott/repos/tam-main/.venv/lib/python3.11/site-packages/snowflake/connector/config_manager.py:369: UserWarning: Bad owner or permissions on /Users/edscott/.snowflake/connections.toml.
 * To change owner, run `chown $USER "/Users/edscott/.snowflake/connections.toml"`.
 * To restrict permissions, run `chmod 0600 "/Users/edscott/.snowflake/connections.toml"`.
 * To skip this warning, set environment variable SF_SKIP_WARNING_FOR_READ_PERMISSIONS_ON_CONFIG_FILE=true.

  warn(f"Bad owner or permissions on {str(filep)}{chmod_message}")
 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://snowbiz.okta.com/app/snowflake/exk8wfsfryJIn4IWZ2p7/sso/saml?SAMLRequest=jVJdc9owEPwrHvUZS3YIJRog44TSOCHAgAkd3oQtHBVZcnVyDP31lfnopA%2FJ9E1z2r3du73e7b6Q3hs3ILTqo8AnyOMq1ZlQeR8tk1GrizywTGVMasX76MAB3Q56wApZ0qiyr2rOf1UcrOcaKaDNRx9VRlHNQABVrOBAbUoX0fOYhj6hDIAb6%2BTQmZKBcFqv1pYU47qu%2FfrK1ybHISEEkxvsUA3kC3onUX6uURptdarlhbJ3M30gEWDSbiQcwinMzsQ7oU4r%2BExlcwIBfUiSWWs2XSTIiy7T3WsFVcHNgps3kfLlfHwyAM7BYjJdPUyXi28%2BKF1vJdvxVBdlZV03373wlmdY6ly4HcXDPip3Ivse%2FJTrg3jskNW4zjtPh66KqydZ3JG2ZpNoNNcveZVv4h9Ep8h7uSQaNonGABWPVZOjdSUSdlokaIU3SRDQq4CG1%2F51p71G3tDlKBSzR%2BbFbGNxI377emfZ0RwrS%2FzXN%2Bb7XbfewtYcHmPVjlfrsPyKATRuYkKnS6FHA2bwv%2FP38HvW%2Bdgmbv%2FxcKalSA%2FeSJuC2Y%2FjCfzgWBFZa3uEUl4wIaMsMxzAxSSlru8NZ9bdtDUVR3hwUv33qgd%2FAA%3D%3D&RelayState=ver%3A1-hint%3A2051115165194-ETMsDgAAAZwJhbz8ABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAECRvZODleDLEfrP3uHynW7YAAACwZ%2BAiBHVVZd4nMssHepRrUjlSXI3Sw2VPUgZ0

 pip install snowflake-connector-python[secure-local-storage]


---
# Key Topic 1: MAXIMIZE

**Get the most out of your queries and data.**

| Sub-Topic | DDM | Source |
|-----------|-----|--------|
| Query Optimization | Partial | Manual |
| Clustering | Yes | TABLE_ETL_V |
| Materialized Views | Yes | TABLE_ETL_V |
| Result Caching | No | Manual |

In [3]:
# Clustering Assessment
cluster_df = session.sql(f"""
SELECT COUNT(*) as total_tables,
    COUNT(CASE WHEN CLUSTER_BY_KEYS IS NOT NULL THEN 1 END) as clustered_tables
FROM {DATABASE}.{SCHEMA}.TABLE_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL AND KIND_ID = 1
""").to_pandas()
cluster_count = cluster_df['CLUSTERED_TABLES'].iloc[0]
print(f"=== Clustering ===\nClustered Tables: {cluster_count}")
cluster_score = 5 if cluster_count >= 20 else 4 if cluster_count >= 10 else 3 if cluster_count >= 5 else 2 if cluster_count >= 1 else 1
print(f"Score: {cluster_score}/5")

=== Clustering ===
Clustered Tables: 1216
Score: 5/5


In [4]:
# Materialized Views
mv_df = session.sql(f"""
SELECT COUNT(*) as mv_count
FROM {DATABASE}.{SCHEMA}.TABLE_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL AND KIND_ID = 7
""").to_pandas()
mv_count = mv_df['MV_COUNT'].iloc[0]
print(f"=== Materialized Views ===\nMVs: {mv_count}")
mv_score = 5 if mv_count >= 10 else 4 if mv_count >= 5 else 3 if mv_count >= 2 else 2 if mv_count >= 1 else 1
print(f"Score: {mv_score}/5")

=== Materialized Views ===
MVs: 0
Score: 1/5


## Manual Assessment: Query Optimization

**Questions:**
1. SELECT * avoided?
2. Query Profile used?
3. Filters on clustered columns?

**Scoring:** 5=Optimized+profiled | 3=Some optimization | 1=No optimization

**Manual Score:** _____

## Manual Assessment: Result Caching

**Questions:**
1. Non-deterministic functions avoided?
2. Caching strategy documented?
3. Cache hit rates monitored?

**Scoring:** 5=Optimized for caching | 3=Some awareness | 1=No awareness

**Manual Score:** _____

In [5]:
query_opt_score = 3; cache_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 1: MAXIMIZE - Summary\n" + "="*60)
print(f"  Query Optimization (Manual): {query_opt_score}/5\n  Clustering (DDM): {cluster_score}/5")
print(f"  Materialized Views (DDM): {mv_score}/5\n  Result Caching (Manual): {cache_score}/5")
maximize_score = round((query_opt_score + cluster_score + mv_score + cache_score) / 4, 1)
print(f"\n📊 MAXIMIZE TOPIC SCORE: {maximize_score}/5")

KEY TOPIC 1: MAXIMIZE - Summary
  Query Optimization (Manual): 3/5
  Clustering (DDM): 5/5
  Materialized Views (DDM): 1/5
  Result Caching (Manual): 3/5

📊 MAXIMIZE TOPIC SCORE: 3.0/5


---
# Key Topic 2: SCALE

**Handle growing workloads efficiently.**

| Sub-Topic | DDM | Source |
|-----------|-----|--------|
| Warehouse Sizing | Yes | WAREHOUSE_ETL_V |
| Multi-Cluster Warehouses | Yes | WAREHOUSE_ETL_V |
| Workload Isolation | Yes | WAREHOUSE_ETL_V |
| Scaling Policies | Yes | WAREHOUSE_ETL_V |

In [6]:
# Warehouse Analysis
wh_df = session.sql(f"""
SELECT COUNT(*) as total_wh,
    COUNT(CASE WHEN MAX_CLUSTER_COUNT > 1 THEN 1 END) as mcw_count,
    COUNT(DISTINCT SIZE) as size_variety
FROM {DATABASE}.{SCHEMA}.WAREHOUSE_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
wh_count = wh_df['TOTAL_WH'].iloc[0]
mcw_count = wh_df['MCW_COUNT'].iloc[0]
print(f"=== Warehouse Scaling ===\nTotal Warehouses: {wh_count}\nMulti-Cluster: {mcw_count}\nSize Variety: {wh_df['SIZE_VARIETY'].iloc[0]}")
wh_score = 5 if wh_count >= 10 else 4 if wh_count >= 5 else 3 if wh_count >= 3 else 2
mcw_score = 5 if mcw_count >= 5 else 4 if mcw_count >= 3 else 3 if mcw_count >= 1 else 1
print(f"Warehouse Score: {wh_score}/5 | MCW Score: {mcw_score}/5")

=== Warehouse Scaling ===
Total Warehouses: 157
Multi-Cluster: 153
Size Variety: 4
Warehouse Score: 5/5 | MCW Score: 5/5


In [7]:
print("="*60 + "\nKEY TOPIC 2: SCALE - Summary\n" + "="*60)
print(f"  Warehouse Sizing: {wh_score}/5\n  Multi-Cluster: {mcw_score}/5\n  Workload Isolation: {wh_score}/5\n  Scaling Policies: {mcw_score}/5")
scale_score = round((wh_score + mcw_score + wh_score + mcw_score) / 4, 1)
print(f"\n📊 SCALE TOPIC SCORE: {scale_score}/5")

KEY TOPIC 2: SCALE - Summary
  Warehouse Sizing: 5/5
  Multi-Cluster: 5/5
  Workload Isolation: 5/5
  Scaling Policies: 5/5

📊 SCALE TOPIC SCORE: 5.0/5


---
# Key Topic 3: IMPROVE

**Continuously monitor and tune performance.**

| Sub-Topic | DDM | Manual |
|-----------|-----|--------|
| Query Profiling | No | Yes |
| Query History Analysis | No | Yes |
| Warehouse Utilization | No | Yes |
| Iterative Tuning | No | Yes |

## Manual Assessment: Query Profiling

**Questions:**
1. Query Profile used regularly?
2. Spillage issues addressed?
3. Profile review cadence?

**Scoring:** 5=Regular profiling+action | 3=Occasional | 1=Never used

**Manual Score:** _____

## Manual Assessment: Query History Analysis

**Questions:**
1. Performance trends monitored?
2. Slow query alerts?
3. Regression detection?

**Scoring:** 5=Proactive monitoring | 3=Reactive | 1=No monitoring

**Manual Score:** _____

## Manual Assessment: Warehouse Utilization

**Questions:**
1. Utilization tracked?
2. Right-sizing reviews?
3. Auto-suspend optimized?

**Scoring:** 5=Optimized+monitored | 3=Basic tracking | 1=No tracking

**Manual Score:** _____

## Manual Assessment: Iterative Tuning

**Questions:**
1. Performance baselines established?
2. A/B testing for changes?
3. Tuning documented?

**Scoring:** 5=Systematic tuning | 3=Ad-hoc tuning | 1=No tuning

**Manual Score:** _____

In [8]:
profiling_score = 3; history_score = 3; util_score = 3; tuning_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 3: IMPROVE - Summary\n" + "="*60)
print(f"  Query Profiling (Manual): {profiling_score}/5\n  Query History (Manual): {history_score}/5")
print(f"  Warehouse Utilization (Manual): {util_score}/5\n  Iterative Tuning (Manual): {tuning_score}/5")
improve_score = round((profiling_score + history_score + util_score + tuning_score) / 4, 1)
print(f"\n📊 IMPROVE TOPIC SCORE: {improve_score}/5")

KEY TOPIC 3: IMPROVE - Summary
  Query Profiling (Manual): 3/5
  Query History (Manual): 3/5
  Warehouse Utilization (Manual): 3/5
  Iterative Tuning (Manual): 3/5

📊 IMPROVE TOPIC SCORE: 3.0/5


---
# Key Topic 4: LEVERAGE

**Use advanced Snowflake capabilities.**

| Sub-Topic | DDM | Source |
|-----------|-----|--------|
| Search Optimization | Yes | TABLE_ETL_V |
| Unistore (Hybrid Tables) | Yes | TABLE_ETL_V |
| Snowpark | Partial | FUNCTION_ETL_V |
| Dynamic Tables | Yes | TABLE_ETL_V |

In [9]:
# Advanced Features
adv_df = session.sql(f"""
SELECT
    COUNT(CASE WHEN KIND_ID = 13 THEN 1 END) as dynamic_tables,
    COUNT(CASE WHEN KIND_ID = 21 THEN 1 END) as hybrid_tables
FROM {DATABASE}.{SCHEMA}.TABLE_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
dt_count = adv_df['DYNAMIC_TABLES'].iloc[0]
hybrid_count = adv_df['HYBRID_TABLES'].iloc[0]
print(f"=== Advanced Features ===\nDynamic Tables: {dt_count}\nHybrid Tables: {hybrid_count}")
dt_score = 5 if dt_count >= 20 else 4 if dt_count >= 10 else 3 if dt_count >= 5 else 2 if dt_count >= 1 else 1
hybrid_score = 5 if hybrid_count >= 5 else 4 if hybrid_count >= 2 else 3 if hybrid_count >= 1 else 1
print(f"DT Score: {dt_score}/5 | Hybrid Score: {hybrid_score}/5")

=== Advanced Features ===
Dynamic Tables: 1060
Hybrid Tables: 0
DT Score: 5/5 | Hybrid Score: 1/5


In [11]:
# Snowpark Usage (LANGUAGE 5 = Python)
sp_df = session.sql(f"""
SELECT COUNT(*) as python_funcs
FROM {DATABASE}.{SCHEMA}.FUNCTION_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL AND LANGUAGE = 5
""").to_pandas()
sp_count = sp_df['PYTHON_FUNCS'].iloc[0]
print(f"=== Snowpark ===\nPython UDFs/Procs: {sp_count}")
sp_score = 5 if sp_count >= 10 else 4 if sp_count >= 5 else 3 if sp_count >= 1 else 1
print(f"Score: {sp_score}/5")

=== Snowpark ===
Python UDFs/Procs: 717
Score: 5/5


In [12]:
print("="*60 + "\nKEY TOPIC 4: LEVERAGE - Summary\n" + "="*60)
print(f"  Dynamic Tables: {dt_score}/5\n  Hybrid Tables: {hybrid_score}/5\n  Snowpark: {sp_score}/5")
leverage_score = round((dt_score + hybrid_score + sp_score) / 3, 1)
print(f"\n📊 LEVERAGE TOPIC SCORE: {leverage_score}/5")

KEY TOPIC 4: LEVERAGE - Summary
  Dynamic Tables: 5/5
  Hybrid Tables: 1/5
  Snowpark: 5/5

📊 LEVERAGE TOPIC SCORE: 3.7/5


---
# Overall Assessment Summary

In [13]:
print("="*70 + "\nPERFORMANCE EFFICIENCY - OVERALL ASSESSMENT\n" + "="*70)
topics = {"1. Maximize": maximize_score, "2. Scale": scale_score, "3. Improve": improve_score, "4. Leverage": leverage_score}
for topic, score in topics.items():
    bar = "█" * int(score) + "░" * (5 - int(score))
    status = "✅" if score >= 4 else "⚠️" if score >= 3 else "❌"
    print(f"{status} {topic}: {bar} {score}/5")
overall_score = round(sum(topics.values()) / len(topics), 1)
print(f"\n🏆 OVERALL PERFORMANCE SCORE: {overall_score}/5")

PERFORMANCE EFFICIENCY - OVERALL ASSESSMENT
⚠️ 1. Maximize: ███░░ 3.0/5
✅ 2. Scale: █████ 5.0/5
⚠️ 3. Improve: ███░░ 3.0/5
⚠️ 4. Leverage: ███░░ 3.7/5

🏆 OVERALL PERFORMANCE SCORE: 3.7/5


In [14]:
prompt = f"""WAF Performance advisor. Provide:
1. EXECUTIVE SUMMARY (2-3 sentences)
2. CRITICAL FINDINGS (3-5 issues with priority)
3. QUICK WINS (3-5 actions for 1 week)

Performance: {overall_score}/5 | Topics: {topics}"""
result = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-sonnet', '{prompt.replace(chr(39), chr(39)+chr(39))}') as response").collect()
print("🤖 AI INSIGHTS\n" + result[0]['RESPONSE'])

🤖 AI INSIGHTS
WAF PERFORMANCE ADVISORY REPORT

1. EXECUTIVE SUMMARY
The Web Application Firewall (WAF) system shows above-average performance (3.7/5) with particularly strong scaling capabilities but requires optimization in rule processing and resource utilization. While the infrastructure demonstrates resilience, there are opportunities to enhance request handling efficiency and reduce false positives without compromising security.

2. CRITICAL FINDINGS
Priority 1: Rule processing overhead causing 23% increased latency in peak hours
Priority 2: Resource utilization imbalance across WAF nodes (72% vs. 45%)
Priority 3: Custom rule conflicts leading to false positives (~8% of legitimate traffic)
Priority 4: Suboptimal caching configuration reducing throughput by ~15%

3. QUICK WINS
a) Implement rule optimization by consolidating overlapping rules and removing redundant conditions (1-2 days)
b) Enable intelligent caching for static content and frequently accessed resources (2-3 days)
c) 

---
## Coverage Matrix

| Topic | Sub-Topic | DDM | Manual | Source |
|-------|-----------|-----|--------|--------|
| Maximize | Query Optimization |
- | ✅ | N/A |
| Maximize | Clustering | ✅ |
- | TABLE_ETL_V |
| Maximize | Materialized Views | ✅ |
- | TABLE_ETL_V |
| Maximize | Result Caching |
- | ✅ | N/A |
| Scale | Warehouse Sizing | ✅ |
- | WAREHOUSE_ETL_V |
| Scale | Multi-Cluster | ✅ |
- | WAREHOUSE_ETL_V |
| Scale | Workload Isolation | ✅ |
- | WAREHOUSE_ETL_V |
| Scale | Scaling Policies | ✅ |
- | WAREHOUSE_ETL_V |
| Improve | Query Profiling |
- | ✅ | N/A |
| Improve | Query History |
- | ✅ | N/A |
| Improve | Warehouse Utilization |
- | ✅ | N/A |
| Improve | Iterative Tuning |
- | ✅ | N/A |
| Leverage | Dynamic Tables | ✅ |
- | TABLE_ETL_V |
| Leverage | Hybrid Tables | ✅ |
- | TABLE_ETL_V |
| Leverage | Snowpark | ✅ |
- | FUNCTION_ETL_V |

**Summary:** 9 DDM | 6 Manual Only